In [1]:
from Bio import SeqIO
import pandas as pd
from src import database, hmm
from Bio.SeqFeature import SeqFeature

In [2]:
accessions = pd.read_csv('data/cox2_th_test_sim/round2/primer-trim-new/new_acc.txt', header=None)[0]
accessions

0      JF793557.1
1     NC_022178.1
2      KF226724.1
3      KM819519.1
4      KM819518.1
5      KM819517.1
6      KM819516.1
7      KM819515.1
8      KM819514.1
9      KM819513.1
10     KM819512.1
11     KM819511.1
12     KM819510.1
13     AF290309.1
14     AF290310.1
15     AF290311.1
16     KC145166.1
17     KJ595384.1
18     KJ595414.1
19     DQ365712.1
20     DQ365759.1
21     DQ365757.1
22     DQ365754.1
23     KX228130.2
24     KX228129.2
25     KX228128.2
26     MF359560.1
27     MF838769.1
28     KY986682.1
29     KY986681.1
30     DQ365705.1
31     DQ365769.1
32     PQ384504.1
33     PQ384503.1
34     PQ384502.1
35     PQ384501.1
36     PQ384500.1
37     PQ384499.1
38     OK185343.1
39     MK455861.1
40     MK455860.1
41     MK455859.1
42     OM317763.1
43     AY913800.1
44     HQ721468.1
45     EF421185.1
46     EF406091.1
47     AF086693.1
Name: 0, dtype: str

In [3]:
gb_records = [record for record in SeqIO.parse('data/cox2_th_test_sim/round2/input-sequences.gb', format='genbank') if record.id in accessions.values]
gb_records

[SeqRecord(seq=Seq('TCCTGCAACTCCTGTTATGGAAGGTATTATAAATTTTCATCATGATTTAATGTA...TAT'), id='JF793557.1', name='JF793557', description='Pustula sp. MT-2011 cytochrome oxidase subunit 2 gene, partial cds; mitochondrial', dbxrefs=[]),
 SeqRecord(seq=Seq('ATATTAATATATTTTATAAAAATTTCAATCAATTTATAACAAAAATTAACGGGA...CAT'), id='NC_022178.1', name='NC_022178', description='Achlya hypogyna mitochondrion, complete genome', dbxrefs=['BioProject:PRJNA927338']),
 SeqRecord(seq=Seq('ATATTAATATATTTTATAAAAATTTCAATCAATTTATAACAAAAATTAACGGGA...CAT'), id='KF226724.1', name='KF226724', description='Achlya hypogyna mitochondrion, complete genome', dbxrefs=[]),
 SeqRecord(seq=Seq('TTAGGATTTTCAGATCCTGCAACCCCAGTTATGGAAGGTATAATTAACTTTCAC...TGG'), id='KM819519.1', name='KM819519', description='Pythiogeton ramosum isolate SC2.7.2 cytochrome c oxidase subunit 2 (COXII) gene, partial cds; mitochondrial', dbxrefs=[]),
 SeqRecord(seq=Seq('TTAGGATTTTCAGATCCTGCAACCCCAGTTATGGAAGGTATAATTAACTTTCAC...TGG'), id='KM819518.1', name=

In [4]:
hmm_hits = hmm.parse_hmm_tblout('data/cox2_th_test_sim/round2/hmm/sequence-scores.tsv')
hmm_hits.dtypes

target_name             str
target_accession        str
query_name              str
query_accession         str
hmm_from              int64
hmm_to                int64
ali_from              int64
ali_to                int64
env_from              int64
env_to                int64
sq_len                int64
strand                  str
e_value             float64
score               float64
bias                float64
description             str
dtype: object

In [5]:
keywords = set()

def is_hit_in_feature(hit: pd.Series, feature: SeqFeature):
    feat_start = int(feature.location.start)
    feat_end = int(feature.location.end)
    return feat_start <= hit['ali_from'] <= feat_end or feat_start <= hit['ali_to'] <= feat_end

for record in gb_records:
    for feature in record.features:
        # Is feature a gene?
        if feature.type != 'CDS':
            continue
        # Find HMM hit location
        hit: pd.Series = hmm_hits.loc[hmm_hits['target_name'] == record.id].iloc[0]
        print(hit)
        if is_hit_in_feature(hit, feature):
            # Add feature name to keyword list
            keywords.update(feature.qualifiers['product'])
keywords

target_name         JF793557.1
target_accession             -
query_name              db-msa
query_accession              -
hmm_from                     1
hmm_to                     288
ali_from                     5
ali_to                     293
env_from                     5
env_to                     293
sq_len                     561
strand                       +
e_value                    0.0
score                    240.6
bias                      61.3
description                  -
Name: 41, dtype: object
target_name         NC_022178.1
target_accession              -
query_name               db-msa
query_accession               -
hmm_from                      1
hmm_to                      288
ali_from                  14895
ali_to                    14607
env_from                  14895
env_to                    14607
sq_len                    46840
strand                        -
e_value                     0.0
score                     217.2
bias                       67.2


{'cytochrome c oxidase subunit 2',
 'cytochrome c oxidase subunit II',
 'cytochrome oxidase subunit 2',
 'cytochrome oxidase subunit II'}

In [5]:
from src import hmm

existing_keywords = [
    'cytochrome c oxidase subunit 2',
    'cytochrome c oxidase subunit ii',
    'cox2',
    'coxii',
    'co2',
    'coii'
]

hmm.find_additional_keywords(hmm_hits, gb_records, existing_keywords)

{'cox ii', 'cytochrome oxidase subunit 2', 'cytochrome oxidase subunit ii'}

In [7]:
from collections.abc import Sequence


def build_search_query(keywords: Sequence[str], organism: str):
    keywords = [f'"{keyword}"' for keyword in keywords]
    joined_keywords = ' OR '.join(keywords)
    # HOTFIX: For reproducability, limit release date
    return f'{joined_keywords} NOT "cytochrome c oxidase subunit 2" NOT "cytochrome c oxidase subunit ii" NOT "cox2" NOT "coxii" NOT "co2" NOT "coii" AND "{organism}"[organism] AND ("0001/01/01"[PDAT] : "2026/04/15"[PDAT])'


build_search_query(keywords, "Oomycota")

'"cytochrome c oxidase subunit II" OR "cytochrome oxidase subunit 2" OR "cytochrome oxidase subunit II" OR "cytochrome c oxidase subunit 2" NOT "cytochrome c oxidase subunit 2" NOT "cytochrome c oxidase subunit ii" NOT "cox2" NOT "coxii" NOT "co2" NOT "coii" AND "Oomycota"[organism] AND ("0001/01/01"[PDAT] : "2026/04/15"[PDAT])'